# 07. Hybrid arm: do contact edges add anything once the nodes carry ESM features?

Question: the structure arm (contact graph, 33 base node features) did not beat random vectors on the cold splits. Here the graph gets frozen per-residue ESM C 300M embeddings as extra node features. Everything else (graph, GCN widths, readout, drug branch, head, training) is the same as the structure arm, so only the node input changes.

Arms:
- `hybrid`: real contact edges + 33 base features + learned 64-d projection of the 960-d residue embedding (97-d node input).
- `hybrid_noedges`: same node features, edges removed (self-loops only). Separates "edges help" from "ESM features help".

Compared against results from the earlier notebooks on the same split: ESM C 300M pooled (PLMNet), the structure arm, the random-vector control, and the drug-mean floor.

Decision rule, fixed before running: `hybrid` counts as better only if its CI mean beats both pooled ESM C 300M and `hybrid_noedges` by more than the 3-seed spread, on the same split. A tie is a valid result.

*Note: The hybrid arm is run on `cold_protein` only. On `cold_family` the structure arm already scores below the drug-mean floor and pooled ESM C 300M is barely above it, so there is no signal there for a hybrid comparison to separate.*

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2. Config

In [ ]:
import os
from glob import glob

def find_folder(name):
    path = f"/content/drive/MyDrive/{name}"
    if not os.path.isdir(path):
        shared = glob(f"/content/drive/.shortcut-targets-by-id/*/{name}")
        if shared:
            path = shared[0]
    assert os.path.isdir(path), f"Folder not found: {path}"
    return path

DRIVE_ROOT = find_folder("Diplomska_hybrid")
COLD_ROOT = find_folder("Diplomska_cold")
PDB_SET = "kinase_domain"
PLM = "esmc_300m"
SPLIT = "cold_protein" # "cold_protein" or "cold_family"; tag part cprot / cfam
STOP_AFTER_FOLDS = None # None = all five folds, 1 = stop once fold 0 is done

os.environ["DTA_DATA_ROOT"] = DRIVE_ROOT
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
os.environ["PYTHONDONTWRITEBYTECODE"] = "1"
print("hybrid folder:", DRIVE_ROOT)
print("cold folder (read only):", COLD_ROOT)
print("split:", SPLIT, "| structure set:", PDB_SET, "| per-residue model:", PLM)

hybrid folder: /content/drive/MyDrive/Diplomska_hybrid
cold folder (read only): /content/drive/MyDrive/Diplomska_cold
split: cold_protein | structure set: kinase_domain | per-residue model: esmc_300m


## 3. GPU

In [ ]:
import torch
print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
!nvidia-smi --query-gpu=name,memory.total --format=csv

Torch version: 2.11.0+cu130
CUDA available: True
name, memory.total [MiB]
Tesla T4, 15360 MiB


## 4. Install packages

In [ ]:
!pip install -q torch_geometric biopython esm

## 5. Go to the folder

In [ ]:
%cd {DRIVE_ROOT}
!pwd

/content/drive/MyDrive/Diplomska_hybrid
/content/drive/MyDrive/Diplomska_hybrid


## 6. Check the uploaded code

In [ ]:
import argparse, json
import numpy as np
import pandas as pd

required = [
    "training_5_folds.py", "testing.py", "report_best_epochs.py", "compute_protein_embeddings.py",
    "src/__init__.py", "src/common/__init__.py", "src/structure/__init__.py", "src/sequence/__init__.py",
    "src/hybrid/__init__.py", "src/hybrid/hybrid_net.py",
    "src/common/seeding.py", "src/common/paths.py", "src/common/run_tag.py", "src/common/split_sets.py",
    "src/common/splits.py", "src/common/structure_sets.py", "src/common/conditions.py",
    "src/common/checkpointing.py", "src/common/dta_dataset.py", "src/common/emetrics.py",
    "src/common/creating_train_and_test_set.py", "src/common/drug_representation.py",
    "src/structure/gnn_net.py", "src/structure/protein_representation.py",
    "src/sequence/plm_net.py", "src/sequence/protein_embeddings.py",
]
missing = [f for f in required if not os.path.exists(f)]
print("Missing code files:", missing or "none")

from src.common.run_tag import add_run_args, run_tag
from src.common.seeding import SEED
parser = argparse.ArgumentParser()
add_run_args(parser)
help_text = parser.format_help()
for word in ["hybrid", "hybrid_noedges", "hybrid_shuffled", "--split", SPLIT]:
    print(f"'{word}' in --help:", word in help_text)

from src.common.paths import paths_for
PATHS = paths_for("davis", PDB_SET, split=SPLIT)
assert PATHS.root == DRIVE_ROOT, f"Data root {PATHS.root} is not {DRIVE_ROOT}"
print("data root:", PATHS.root)
print("train folds file:", PATHS.train_folds, os.path.exists(PATHS.train_folds))
print("test fold file:  ", PATHS.test_fold, os.path.exists(PATHS.test_fold))

TAG_PART = {"cold_family": "cfam", "cold_protein": "cprot"}[SPLIT]
print("Example tags:", run_tag("hybrid", PLM, SEED, False, PDB_SET, split=SPLIT), "|",
      run_tag("hybrid_noedges", PLM, SEED, False, PDB_SET, split=SPLIT), "|",
      run_tag("structure", None, SEED, False, PDB_SET, split=SPLIT))
assert run_tag("structure", None, SEED, False, PDB_SET, split=SPLIT) == f"pdb2_{TAG_PART}", "structure tag changed"
assert run_tag("plm", PLM, SEED, False, PDB_SET, split=SPLIT) == f"{PLM}_pdb2_{TAG_PART}", "PLM tag changed"

Missing code files: none
'hybrid' in --help: True
'hybrid_noedges' in --help: True
'hybrid_shuffled' in --help: True
'--split' in --help: True
'cold_protein' in --help: True
data root: /content/drive/MyDrive/Diplomska_hybrid
train folds file: /content/drive/MyDrive/Diplomska_hybrid/data/davis/splits/cold_protein/train_folds.txt True
test fold file:   /content/drive/MyDrive/Diplomska_hybrid/data/davis/splits/cold_protein/test_fold.txt True
Example tags: hybrid_esmc_300m_pdb2_cprot | hybrid_noedges_esmc_300m_pdb2_cprot | pdb2_cprot


## 7. Copy data from Diplomska_cold

In [ ]:
import shutil, time

SRC_DAVIS = os.path.join(COLD_ROOT, "data", "davis")
DST_DAVIS = os.path.join(DRIVE_ROOT, "data", "davis")
assert os.path.isdir(SRC_DAVIS), SRC_DAVIS
os.makedirs(DST_DAVIS, exist_ok=True)

def copy_tree(src, dst, keep_models=None):
    n_copied = n_skipped = nbytes = 0
    for root, dirs, files in os.walk(src):
        if keep_models is not None and os.path.basename(root).startswith("protein_embeddings"):
            dirs[:] = [d for d in dirs if d in keep_models]
        rel = os.path.relpath(root, src)
        out = dst if rel == "." else os.path.join(dst, rel)
        os.makedirs(out, exist_ok=True)
        for name in files:
            s, d = os.path.join(root, name), os.path.join(out, name)
            size = os.path.getsize(s)
            if os.path.exists(d) and os.path.getsize(d) == size:
                n_skipped += 1
                continue
            shutil.copyfile(s, d)
            n_copied += 1; nbytes += size
    return n_copied, n_skipped, nbytes

KEEP_EMBEDDINGS = {PLM}
t_all = time.time()
for entry in sorted(os.listdir(SRC_DAVIS)):
    s = os.path.join(SRC_DAVIS, entry); d = os.path.join(DST_DAVIS, entry)
    t0 = time.time()
    if os.path.isdir(s):
        c, k, b = copy_tree(s, d, keep_models=KEEP_EMBEDDINGS)
        print(f"{entry:40} copied {c:5} skipped {k:5} {b / 2**20:8.1f} MB  {time.time() - t0:6.0f} s", flush=True)
    else:
        if not (os.path.exists(d) and os.path.getsize(d) == os.path.getsize(s)):
            shutil.copyfile(s, d)
        print(f"{entry:40} file {os.path.getsize(s) / 2**20:8.1f} MB  {time.time() - t0:6.0f} s", flush=True)
print(f"data copied in {(time.time() - t_all) / 60:.1f} min")

REFERENCE_DIR = os.path.join(DRIVE_ROOT, "reference_cold")
os.makedirs(REFERENCE_DIR, exist_ok=True)
os.makedirs(PATHS.models, exist_ok=True)
os.makedirs(PATHS.results, exist_ok=True)
src_results = os.path.join(COLD_ROOT, "results_davis")
n = 0
for name in os.listdir(src_results):
    if name.startswith("result_") and name.endswith(".txt"):
        s, d = os.path.join(src_results, name), os.path.join(REFERENCE_DIR, name)
        if not (os.path.exists(d) and os.path.getsize(d) == os.path.getsize(s)):
            shutil.copyfile(s, d)
        n += 1
print("reference result files:", n)

contact_maps                             copied     0 skipped   442      0.0 MB       6 s
contact_maps_kinase_domain               copied     0 skipped   442      0.0 MB       3 s
davis.csv                                file      2.3 MB       0 s
ligand_graphs                            copied     0 skipped    68      0.0 MB       1 s
ligand_graphs2.json                      file      7.7 MB       0 s
ligand_plots                             copied     0 skipped    68      0.0 MB       1 s
pdb                                      copied     0 skipped   442      0.0 MB       2 s
pdb_kinase_domain                        copied     0 skipped   442      0.0 MB       3 s
protein_embeddings_kinase_domain         copied     0 skipped   430      0.0 MB       4 s
protein_gen.log                          file      0.1 MB       0 s
protein_graphs                           copied     0 skipped   442      0.0 MB       6 s
protein_graphs_kinase_domain             copied     0 skipped   442      0.0

## 8. Check the data and the split

In [ ]:
df = pd.read_csv(PATHS.csv)
names = df["protein"].unique()
pdb_missing = [n for n in names if not os.path.exists(PATHS.pdb_file(n))]
print(f"proteins in pair table: {len(names)} (expected 442)")
print(f"PDB files missing for {len(pdb_missing)} proteins <- expected 0")
print(f"protein graphs: {len(glob(os.path.join(PATHS.protein_graph_gml, '*_contact_map_graph.gml')))} (expected 442)")
print(f"ligand graphs: {len(glob(os.path.join(PATHS.ligand_graphs, '*.gml')))} (expected 68)")
for m in sorted(KEEP_EMBEDDINGS):
    print(f"{m} pooled vectors:", len(glob(os.path.join(PATHS.protein_embeddings, m, "*.pooled.pt"))), "(expected 428)")

def read_indices(path):
    with open(path) as f:
        return json.load(f)

folds = read_indices(PATHS.train_folds)
test_idx = [int(i) for i in read_indices(PATHS.test_fold)]
train_idx = [int(i) for fold in folds for i in fold]
print("train fold sizes:", [len(x) for x in folds], "| train rows:", len(train_idx), "| test rows:", len(test_idx))
assert max(train_idx + test_idx) < len(df)
train_prot = set(df.iloc[train_idx]["protein"]); test_prot = set(df.iloc[test_idx]["protein"])
print("train proteins:", len(train_prot), "| test proteins:", len(test_prot), "(expected 88)")
print("proteins in both train and test:", len(train_prot & test_prot), "<- expected 0")
fold_prot = [set(df.iloc[[int(i) for i in fold]]["protein"]) for fold in folds]
print("proteins shared between two train folds:", sum(len(fold_prot[i] & fold_prot[j]) for i in range(5) for j in range(i + 1, 5)), "<- expected 0")
print("drugs in test:", df.iloc[test_idx]["ligand"].nunique(), "| drugs in train:", df.iloc[train_idx]["ligand"].nunique(), "(expected 68, 68)")
assert not (train_prot & test_prot), "Test proteins leak into training"

proteins in pair table: 442 (expected 442)
PDB files missing for 0 proteins <- expected 0
protein graphs: 442 (expected 442)
ligand graphs: 68 (expected 68)
esmc_300m pooled vectors: 428 (expected 428)
train fold sizes: [4828, 4828, 4828, 4828, 4760] | train rows: 24072 | test rows: 5984
train proteins: 354 | test proteins: 88 (expected 88)
proteins in both train and test: 0 <- expected 0
proteins shared between two train folds: 0 <- expected 0
drugs in test: 68 | drugs in train: 68 (expected 68, 68)


## 9. Helpers

In [ ]:
import re, statistics, subprocess, threading, time

QUIET_PREFIXES = ("Train epoch:", "Training on ", "Predicting for validation data",
                  "Make prediction for", "MSE improved at epoch", "No improvement since epoch")
VALIDATION_LINE = re.compile(r"^Epoch (\d+) - Validation MSE")

def gpu_memory_mb():
    try:
        out = subprocess.check_output(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"], text=True)
        return int(out.strip().splitlines()[0])
    except Exception:
        return None

def run(args, stop_marker=None, every=10):
    cmd = ["python", "-u"] + args
    print("Running:", " ".join(cmd))
    peak = [0]; done = threading.Event()
    def poll():
        while not done.is_set():
            m = gpu_memory_mb()
            if m is not None:
                peak[0] = max(peak[0], m)
            done.wait(20)
    threading.Thread(target=poll, daemon=True).start()
    start = time.time()
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    stopped = False
    try:
        for line in proc.stdout:
            match = VALIDATION_LINE.match(line)
            if match:
                if int(match.group(1)) % every == 0:
                    print(line, end="")
                continue
            if line.startswith(QUIET_PREFIXES):
                continue
            print(line, end="")
            if stop_marker and stop_marker in line:
                stopped = True
                print("\nStopping here as requested. Rerun this cell to continue; finished folds are skipped.")
                break
    finally:
        if proc.poll() is None:
            proc.terminate()
        proc.wait()
        done.set()
    print(f"[elapsed {(time.time() - start) / 60:.1f} min | peak GPU memory seen {peak[0]} MB (nvidia-smi, sampled every 20 s)]")
    if not stopped and proc.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {proc.returncode}, see the output above.")

ARMS = {
    "hybrid": ("hybrid", "HybridNet", PLM, "new"),
    "hybrid_noedges": ("hybrid_noedges", "HybridNet", PLM, "new"),
    "hybrid_shuffled": ("hybrid_shuffled", "HybridNet", PLM, "new"),
    PLM: ("plm", "PLMNet", PLM, "ref"),
    "structure": ("structure", "GNNNet", None, "ref"),
    "random_1024": ("plm", "PLMNet", "random_1024", "ref"),
}
NEW_ARMS = [a for a, v in ARMS.items() if v[3] == "new"]

def flags(arm, seed=SEED):
    protein_repr, _, plm, _ = ARMS[arm]
    out = ["--protein-repr", protein_repr, "--pdb-set", PDB_SET, "--split", SPLIT]
    if plm:
        out += ["--plm-model", plm]
    if seed != SEED:
        out += ["--seed", str(seed)]
    return out

def arm_tag(arm, seed=SEED):
    protein_repr, _, plm, _ = ARMS[arm]
    tag = run_tag(protein_repr, plm, seed, False, PDB_SET, split=SPLIT)
    assert TAG_PART in tag, tag
    return tag

def stop_marker():
    return f"Training for fold {STOP_AFTER_FOLDS}..." if STOP_AFTER_FOLDS else None

def results_root(arm):
    return PATHS.results if ARMS[arm][3] == "new" else REFERENCE_DIR

def result_path(arm, seed=SEED, full=False):
    return os.path.join(results_root(arm), f"result_{ARMS[arm][1]}_davis_{arm_tag(arm, seed)}{'_full' if full else ''}.txt")

def model_path(arm, seed=SEED):
    return os.path.join(PATHS.models, f"model_{ARMS[arm][1]}_davis_{arm_tag(arm, seed)}_full.model")

def median_best_epoch(arm):
    path = os.path.join(PATHS.results, f"best_epochs_{arm_tag(arm)}.json")
    with open(path) as f:
        records = json.load(f)
    fold_ids = [str(i) for i in range(5)]
    absent = [i for i in fold_ids if i not in records]
    assert not absent, f"Folds {absent} are not finished yet in {path}"
    print(f"{'fold':>4} {'best_epoch':>10} {'last_epoch':>10} {'best_val_mse':>13}")
    for i in fold_ids:
        r = records[i]
        print(f"{i:>4} {r['best_epoch']:>10} {r['last_epoch']:>10} {r['best_val_mse']:>13.4f}")
    median = int(round(statistics.median(records[i]["best_epoch"] for i in fold_ids)))
    print("Median best epoch (use for the full-data model):", median)
    return median

def show(path):
    print(path)
    with open(path) as f:
        print(f.read())

def read_mean(path, metric):
    if not os.path.exists(path):
        return None
    with open(path) as f:
        for line in f:
            if line.startswith(metric + " ") and "mean:" in line:
                return float(line.split("mean:")[1].split()[0])
    return None

def cv_stage(arm):
    assert arm in NEW_ARMS, arm
    print("=" * 70); print(f"{arm}: five folds on {SPLIT}, tag {arm_tag(arm)}")
    run(["training_5_folds.py"] + flags(arm), stop_marker=stop_marker())
    if STOP_AFTER_FOLDS:
        print("Partial run (STOP_AFTER_FOLDS is set): set it to None and rerun this cell to finish the folds.")
        return
    median_best_epoch(arm)
    run(["testing.py"] + flags(arm))
    show(result_path(arm))

def full_stage(arm, seed=SEED):
    assert arm in NEW_ARMS, arm
    epochs = median_best_epoch(arm)
    print("=" * 70); print(f"{arm}: full-data model, seed {seed}, {epochs} epochs, tag {arm_tag(arm, seed)}")
    if os.path.exists(model_path(arm, seed)):
        print("Model already finished, skipping training:", model_path(arm, seed))
    else:
        run(["training_5_folds.py"] + flags(arm, seed) + ["--mode", "full", "--epochs", str(epochs)], every=50)
    if os.path.exists(result_path(arm, seed, full=True)):
        print("Result already exists, skipping test:", result_path(arm, seed, full=True))
    else:
        run(["testing.py"] + flags(arm, seed) + ["--full"])
    show(result_path(arm, seed, full=True))

# Baselines without protein information
The split's own training rows and test rows are used. A test protein is never in the training rows, so the best no-protein predictor is the mean of its drug. The arms must beat this to claim they learned anything about proteins.

In [ ]:
def concordance(y, p, chunk=500):
    y = np.asarray(y, dtype=np.float64); p = np.asarray(p, dtype=np.float64)
    num = 0.0; den = 0
    for i in range(0, len(y), chunk):
        dy = y[i:i + chunk, None] - y[None, :]
        dp = p[i:i + chunk, None] - p[None, :]
        pairs = dy > 0
        num += float((pairs & (dp > 0)).sum()) + 0.5 * float((pairs & (dp == 0)).sum())
        den += int(pairs.sum())
    return num / den

train = df.iloc[train_idx]; test = df.iloc[test_idx]
y_test = test["label"].to_numpy()
mu = train["label"].mean()
drug_mean = train.groupby("ligand")["label"].mean()
prot_mean = train.groupby("protein")["label"].mean()

predictions = {
    "global mean": np.full(len(test), mu),
    "drug mean": test["ligand"].map(drug_mean).fillna(mu).to_numpy(),
    "drug mean + protein mean (protein unseen)": (test["ligand"].map(drug_mean).fillna(mu)
                                                  + test["protein"].map(prot_mean).fillna(mu) - mu).to_numpy(),
}
print(f"split {SPLIT}: test rows {len(test)}, test proteins with a training mean: "
      f"{int(test['protein'].isin(train['protein']).sum())} rows (expected 0)")
print(f"{'predictor':45} {'MSE':>8} {'CI':>8}")
BASELINE = {}
for name, pred in predictions.items():
    mse = float(np.mean((y_test - pred) ** 2)); ci = concordance(y_test, pred)
    BASELINE[name] = (mse, ci)
    print(f"{name:45} {mse:8.4f} {ci:8.4f}")

split cold_protein: test rows 5984, test proteins with a training mean: 0 rows (expected 0)
predictor                                          MSE       CI
global mean                                     0.7806   0.5000
drug mean                                       0.6288   0.7470
drug mean + protein mean (protein unseen)       0.6288   0.7470


# Per-residue embeddings
Computes ESM C 300M embeddings per residue for the 428 unique kinase-domain sequences and stores them as float16 (about 228 MB in total).

In [ ]:
run(["compute_protein_embeddings.py", "--model", PLM, "--pdb-set", PDB_SET, "--keep-residues"])
cache = os.path.join(PATHS.protein_embeddings, PLM)
files = glob(os.path.join(cache, "*"))
total = sum(os.path.getsize(f) for f in files if os.path.isfile(f))
print("files in", cache, ":", len(files), "| total size", round(total / 2**20), "MB")
print("example names:", sorted(os.path.basename(f) for f in files)[:4])

Running: python -u compute_protein_embeddings.py --model esmc_300m --pdb-set kinase_domain --keep-residues
Structures: kinase_domain (/content/drive/MyDrive/Diplomska_hybrid/data/davis/pdb_kinase_domain)
442 proteins, 428 unique sequences, 428 to embed with esmc_300m
Loading esmc_300m on cuda
ESMC: Transformer Engine is not installed; falling back to pure-PyTorch LayerNorm+Linear / LayerNorm+MLP. Outputs will differ numerically - measured on the unnormalized residual stream (before the final LayerNorm), ~O(10) max-diff in fp32 and ~O(100) in bf16; after the final LayerNorm these shrink to a few ULP and perplexity stays within rounding noise. Install with `pip install transformer-engine[pytorch]` to enable fused fp32-reduction LayerNorm.
ESMC: neither xformers nor flash-attn is installed; falling back to PyTorch ``F.scaled_dot_product_attention``. The attention reduction order in bf16 differs from a fused kernel by ~1 bf16 ULP per attention block; compounded across the 80-block stack th

# Arms

## hybrid: five folds

In [ ]:
cv_stage("hybrid")

hybrid: five folds on cold_protein, tag hybrid_esmc_300m_pdb2_cprot
Running: python -u training_5_folds.py --protein-repr hybrid --pdb-set kinase_domain --split cold_protein --plm-model esmc_300m
Dataset:  davis
Mode:  cv
Protein representation:  hybrid
Structure set:  kinase_domain
Protein language model:  esmc_300m
Standardized embeddings:  False
Seed:  42
Learning rate:  0.001
Epochs:  2000
Early stop patience:  100
Data root:  /content/drive/MyDrive/Diplomska_hybrid
Using device: cuda
Run tag:  hybrid_esmc_300m_pdb2_cprot
Loaded per-residue esmc_300m embeddings for 442 proteins, dim 960, all aligned with the graph nodes
Fold 0 already finished at epoch 620 (best MSE 0.3749765455722809 at epoch 520), skipping
Resuming fold 1 at epoch 501 (best MSE 0.28879064321517944 at epoch 406)
Training for fold 1...
/content/drive/MyDrive/Diplomska_hybrid/training_5_folds.py:42: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  return DataLoader(
/content/drive/MyDr

## hybrid: full data

In [ ]:
full_stage("hybrid")

fold best_epoch last_epoch  best_val_mse
   0        520        620        0.3750
   1        406        506        0.2888
   2        411        511        0.4221
   3        408        508        0.3571
   4        260        360        0.3608
Median best epoch (use for the full-data model): 408
hybrid: full-data model, seed 42, 408 epochs, tag hybrid_esmc_300m_pdb2_cprot
Running: python -u training_5_folds.py --protein-repr hybrid --pdb-set kinase_domain --split cold_protein --plm-model esmc_300m --mode full --epochs 408
Dataset:  davis
Mode:  full
Protein representation:  hybrid
Structure set:  kinase_domain
Protein language model:  esmc_300m
Standardized embeddings:  False
Seed:  42
Learning rate:  0.001
Epochs:  408
Data root:  /content/drive/MyDrive/Diplomska_hybrid
Using device: cuda
Run tag:  hybrid_esmc_300m_pdb2_cprot
Loaded per-residue esmc_300m embeddings for 442 proteins, dim 960, all aligned with the graph nodes
/content/drive/MyDrive/Diplomska_hybrid/training_5_folds.py

## hybrid_noedges: five folds

In [ ]:
cv_stage("hybrid_noedges")

hybrid_noedges: five folds on cold_protein, tag hybrid_noedges_esmc_300m_pdb2_cprot
Running: python -u training_5_folds.py --protein-repr hybrid_noedges --pdb-set kinase_domain --split cold_protein --plm-model esmc_300m
Dataset:  davis
Mode:  cv
Protein representation:  hybrid_noedges
Structure set:  kinase_domain
Protein language model:  esmc_300m
Standardized embeddings:  False
Seed:  42
Learning rate:  0.001
Epochs:  2000
Early stop patience:  100
Data root:  /content/drive/MyDrive/Diplomska_hybrid
Using device: cuda
Run tag:  hybrid_noedges_esmc_300m_pdb2_cprot
Loaded per-residue esmc_300m embeddings for 442 proteins, dim 960, all aligned with the graph nodes
Resuming fold 0 at epoch 801 (best MSE 0.37251949310302734 at epoch 705)
Training for fold 0...
/content/drive/MyDrive/Diplomska_hybrid/training_5_folds.py:42: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  return DataLoader(
/content/drive/MyDrive/Diplomska_hybrid/training_5_folds.py:114: User

## hybrid_noedges: full data

In [ ]:
full_stage("hybrid_noedges")

fold best_epoch last_epoch  best_val_mse
   0        705        805        0.3725
   1        586        686        0.2830
   2        457        557        0.4030
   3        331        431        0.3771
   4        150        250        0.3925
Median best epoch (use for the full-data model): 457
hybrid_noedges: full-data model, seed 42, 457 epochs, tag hybrid_noedges_esmc_300m_pdb2_cprot
Running: python -u training_5_folds.py --protein-repr hybrid_noedges --pdb-set kinase_domain --split cold_protein --plm-model esmc_300m --mode full --epochs 457
Dataset:  davis
Mode:  full
Protein representation:  hybrid_noedges
Structure set:  kinase_domain
Protein language model:  esmc_300m
Standardized embeddings:  False
Seed:  42
Learning rate:  0.001
Epochs:  457
Data root:  /content/drive/MyDrive/Diplomska_hybrid
Using device: cuda
Run tag:  hybrid_noedges_esmc_300m_pdb2_cprot
Loaded per-residue esmc_300m embeddings for 442 proteins, dim 960, all aligned with the graph nodes
/content/drive/MyDr

# Seeds 43 and 44 of the full-data models
Same protocol as the earlier arms: the median best epoch comes from the seed-42 five folds, only the full-data model is retrained with another seed.

In [ ]:
SEEDS = [43, 44]
SEED_ARMS = ["hybrid", "hybrid_noedges"]
for arm in SEED_ARMS:
    for seed in SEEDS:
        full_stage(arm, seed)

fold best_epoch last_epoch  best_val_mse
   0        520        620        0.3750
   1        406        506        0.2888
   2        411        511        0.4221
   3        408        508        0.3571
   4        260        360        0.3608
Median best epoch (use for the full-data model): 408
hybrid: full-data model, seed 43, 408 epochs, tag hybrid_esmc_300m_pdb2_cprot_s43
Model already finished, skipping training: /content/drive/MyDrive/Diplomska_hybrid/models_davis/model_HybridNet_davis_hybrid_esmc_300m_pdb2_cprot_s43_full.model
Result already exists, skipping test: /content/drive/MyDrive/Diplomska_hybrid/results_davis/result_HybridNet_davis_hybrid_esmc_300m_pdb2_cprot_s43_full.txt
/content/drive/MyDrive/Diplomska_hybrid/results_davis/result_HybridNet_davis_hybrid_esmc_300m_pdb2_cprot_s43_full.txt
davis  folds evaluated: ['full']
ci (all pairs) is the value to compare with published numbers; cindex only counts pairs whose larger label has the larger row index.

fold full  rmse: 

# Summary

In [ ]:
SEED_LIST = [SEED, 43, 44]
floor_mse, floor_ci = BASELINE["drug mean"]

def seed_values(arm, metric):
    vals = [read_mean(result_path(arm, s, full=True), metric) for s in SEED_LIST]
    return [v for v in vals if v is not None]

print(f"split: {SPLIT} | drug-mean floor: MSE {floor_mse:.4f}, CI {floor_ci:.4f} | decision uses CI")
print(f"{'arm':20} {'n':>2} {'CI mean':>8} {'CI std':>7} {'CI min':>7} {'above floor':>12} {'gain over floor':>16} {'MSE median':>11} {'MSE max':>8}")
STATS = {}
for arm in ARMS:
    ci = seed_values(arm, "ci"); mse = seed_values(arm, "mse")
    if not ci:
        print(f"{arm:20} {'-':>2}")
        continue
    sd = statistics.stdev(ci) if len(ci) > 1 else float("nan")
    STATS[arm] = (statistics.mean(ci), sd)
    above = sum(c > floor_ci for c in ci)
    print(f"{arm:20} {len(ci):>2} {statistics.mean(ci):8.4f} {sd:7.4f} {min(ci):7.4f} {above:>9}/{len(ci)} "
          f"{statistics.mean(ci) - floor_ci:+16.4f} {statistics.median(mse):11.4f} {max(mse):8.4f}")

print()
print("Decision rule: hybrid beats both comparisons by more than the larger seed std of the pair.")
for other in [PLM, "hybrid_noedges"]:
    if "hybrid" in STATS and other in STATS:
        diff = STATS["hybrid"][0] - STATS[other][0]
        spread = max(STATS["hybrid"][1], STATS[other][1])
        verdict = "beyond spread" if diff > spread else ("within spread (tie)" if abs(diff) <= spread else "worse beyond spread")
        print(f"  hybrid - {other:16}: {diff:+.4f}  | spread {spread:.4f}  -> {verdict}")

split: cold_protein | drug-mean floor: MSE 0.6288, CI 0.7470 | decision uses CI
arm                   n  CI mean  CI std  CI min  above floor  gain over floor  MSE median  MSE max
hybrid                3   0.7960  0.0069  0.7885         3/3          +0.0489      0.5923   0.6201
hybrid_noedges        3   0.8000  0.0061  0.7953         3/3          +0.0529      0.5452   0.5814
hybrid_shuffled       -
esmc_300m             3   0.8265  0.0060  0.8224         3/3          +0.0795      0.4603   0.4913
structure             3   0.7216  0.0031  0.7187         0/3          -0.0254      0.7034   0.7560
random_1024           3   0.7233  0.0051  0.7176         0/3          -0.0237      0.7372   1.0010

Decision rule: hybrid beats both comparisons by more than the larger seed std of the pair.
  hybrid - esmc_300m       : -0.0305  | spread 0.0069  -> worse beyond spread
  hybrid - hybrid_noedges  : -0.0040  | spread 0.0069  -> within spread (tie)
